# RQ2: E/random selection × merge/no-merge

Use Internet and **T4 ×2**, the same original locked runtime and the existing private **rq1-ih-inputs-v1** Dataset. Push the new files first.

Start with the **seven-question crossed calibration**. It compares full576, native E merge/no-merge, and random merge/no-merge at three fixed seeds: **nine conditions per question**. Every compressed condition keeps 288 states and the same ten native FPS basis source-to-spatial-slot assignments. Only the other 278 selected positions change. Compact basis positions may change when the selected subset changes.

Three random seeds are repeated conditions on the same questions, not three independent datasets. All original model files, E/FPS/merge math and dependencies stay unchanged; random selection is an explicitly project-owned control. Each image's first captured tuples are replayed unchanged for every same-image question and both generation/scoring. Independent native numerical repeats remain separate controls.

The optional **Brolio oracle** is a separate one-question run. It restores target317, includes a no-op and a matched control, and cannot satisfy the main calibration gate or be presented as a deployable policy. See `research/RQ2_CROSSED.md`.

All flags default to False and prevent accidental setup/inference when imported as a running version. Open an editable draft and enable exactly one mode.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_CROSSED_CALIBRATION = False  # First: seven questions, nine conditions each.
RUN_CROSSED_FULL = False         # Later: full30, after crossed calibration review.
RUN_BROLIO_ORACLE = False        # Optional independent one-question oracle diagnostic.
CALIBRATION_REPORT = None        # Full mode only: crossed report.json with all detail files.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None                 # Same existing rq1-ih-inputs-v1 ZIP or extracted Dataset.
CHECKPOINT_ARGS = []             # Optional existing original-format model/vision paths.

if sum([RUN_CROSSED_CALIBRATION, RUN_CROSSED_FULL, RUN_BROLIO_ORACLE]) != 1:
    raise RuntimeError("Enable exactly one mode; start with RUN_CROSSED_CALIBRATION = True.")
if RUN_CROSSED_FULL and (CALIBRATION_REPORT is None or not Path(CALIBRATION_REPORT).is_file()):
    raise ValueError("Full mode needs the passed crossed calibration report.json and its detailed evidence.")
print("Preparing RQ2 crossed code; model inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
if not (repo / "scripts/run_rq2_crossed.py").is_file():
    raise RuntimeError("Push the new RQ2 crossed files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
rq2_crossed_runs = []
print("Repository:", repo, flush=True)


## Prepare the accepted inputs and original environment

Reuse the same input Dataset. A live session reuses its environment and cached checkpoints; a new notebook session installs the same locked runtime without changing Kaggle's default kernel. Input attachment does not share a virtual environment between notebooks.


In [ ]:
sys.path.insert(0, str(repo / "scripts"))
from rq1_input_ih import locate_inputs
inputs = locate_inputs(
    [Path("/kaggle/input"), Path("/kaggle/working")],
    Path("/kaggle/working/rq1-ih-inputs-v1"), explicit=INPUT_ZIP,
)
print("Accepted input directory:", inputs, flush=True)
setup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
setup_log = Path(f"/kaggle/working/rq2-crossed-setup-{setup_stamp}.log")
runtime_report = Path(f"/kaggle/working/rq2-crossed-runtime-{setup_stamp}.json")

def stream_command(command, stream, *, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    stream.write("COMMAND " + repr(command) + "\n")
    stream.flush()
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    stream.write(f"EXIT {status}\n")
    stream.flush()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); preserve {stream.name} before changing anything.")
    return "".join(lines)

with setup_log.open("x", encoding="utf-8") as stream:
    stream_command([
        "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
        "--output", str(runtime_report),
    ], stream, cwd=repo)
print("Locked original runtime ready. Run the prerequisite/helper cell next.", flush=True)
print("Setup log:", setup_log)
print("Runtime report:", runtime_report)


## Native prerequisites and streaming run helper

Prerequisites include the actual tiny native random-selection/merge pairing, source/basis identity, scoped restoration, replay and oracle checks. Torch must import and no native test may skip. A passed CPU test alone does not establish pretrained behavior.

The helper preserves a runner ZIP on handled failure and a separate console log. If the runner cannot create its ZIP, the wrapper labels its fallback as incomplete evidence. Do not update code or run concurrent model processes during inference.


In [ ]:
def verify_rq2_crossed_prerequisites(*, refresh_runtime=False):
    """Run tiny native tests; refresh provenance without reinstalling in a live session."""
    with setup_log.open("a", encoding="utf-8") as stream:
        if refresh_runtime:
            stream_command([
                str(original_python), "-X", "utf8", "-I", "scripts/check_original_env.py",
                "--require-cuda", "--expected-gpus", "2", "--output", str(runtime_report),
            ], stream, cwd=repo)
        # Import Torch explicitly so Kaggle cannot take the local no-Torch skip path.
        stream_command([str(original_python), "-X", "utf8", "-I", "-c",
                        "import torch; print(torch.__version__)"], stream, cwd=repo)
        native_output = stream_command([
            str(original_python), "-X", "utf8", "-m", "unittest",
            "research.tests.test_rq2_native",
            "research.tests.test_rq2_frozen_native",
            "research.tests.test_rq2_crossed_native", "-v",
        ], stream, cwd=repo)
        count = re.search(r"Ran (\d+) tests?", native_output)
        if not count or int(count.group(1)) == 0 or re.search(r"skipped\s*=", native_output):
            raise RuntimeError("Native prerequisites must run with zero skips; preserve the setup log.")
    print("RQ2 crossed native prerequisites passed.", flush=True)


def run_rq2_crossed(*, mode="calibration", calibration_report=None):
    if mode not in ("calibration", "full", "oracle"):
        raise ValueError("mode must be calibration, full or oracle.")
    if mode != "full" and calibration_report is not None:
        raise ValueError("Only full mode accepts a crossed calibration report.")
    if mode == "full":
        if calibration_report is None or not Path(calibration_report).is_file():
            raise ValueError("Full mode needs the passed crossed calibration report and its details.")
        calibration_report = Path(calibration_report)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    prefix = f"rq2-crossed-{mode}-v1"
    folder = Path(f"/kaggle/working/{prefix}-{stamp}")
    log = folder.with_suffix(".log")
    archive = folder.with_suffix(".zip")
    globals().setdefault("rq2_crossed_runs", []).append(folder)
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_rq2_crossed.py",
        "--inputs", str(inputs), "--output", str(folder),
        "--require-cuda", "--expected-gpus", "2", "--download-budget-gib", "20",
        "--setup-log", str(setup_log), "--runtime-report", str(runtime_report), *CHECKPOINT_ARGS]
    if mode == "calibration":
        command.append("--calibration")
    elif mode == "oracle":
        command.append("--oracle-only")
    else:
        command.extend(["--calibration-report", str(calibration_report)])
    print("Mode:", mode, "(nine-condition crossed comparison; oracle is separate)", flush=True)
    print("Run directory:", folder, flush=True)
    fallback_archive = False
    try:
        with log.open("x", encoding="utf-8") as stream:
            stream_command(command, stream, cwd=repo)
    finally:
        # Normal handled failures already have a complete runner evidence ZIP.
        if not archive.exists():
            fallback_archive = True
            folder.mkdir(parents=True, exist_ok=True)
            fallback = folder / "notebook-fallback"
            fallback.mkdir(exist_ok=True)
            (fallback / "README.txt").write_text(
                "Notebook fallback evidence: the process did not produce its normal ZIP.\n"
                "This archive does not establish successful execution or complete results.\n",
                encoding="utf-8")
            for source in [setup_log, runtime_report, log]:
                if source.is_file():
                    shutil.copy2(source, fallback / source.name)
            with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as bundle:
                for source in sorted(folder.rglob("*")):
                    if source.is_file():
                        bundle.write(source, source.relative_to(folder))
            print("Created incomplete fallback evidence ZIP:", archive, flush=True)
        print("Download:", archive, "and", log, flush=True)
        report_path = folder / "report.json"
        if report_path.exists():
            try:
                report = json.loads(report_path.read_text(encoding="utf-8"))
                print("Status:", report.get("status"), "mode:", report.get("mode"),
                      "wall seconds:", report.get("wall_seconds"), flush=True)
                for name in ["progress", "summaries", "error"]:
                    if report.get(name) is not None:
                        print(name + ":", report[name], flush=True)
            except (ValueError, OSError) as error:
                print("Could not read final report:", error, flush=True)
    if fallback_archive:
        raise RuntimeError("Runner archive missing; preserve the fallback ZIP/log before rerunning.")
    if not (folder / "report.json").is_file():
        raise RuntimeError("No runner report was produced; preserve the ZIP/log.")
    report = json.loads((folder / "report.json").read_text(encoding="utf-8"))
    if report.get("version") != "rq2-crossed-input-v1":
        raise RuntimeError("Unexpected runner version; preserve the ZIP/log.")
    if report.get("mode") != mode:
        raise RuntimeError("Runner mode mismatch; preserve the ZIP/log.")
    if report.get("status") != "passed":
        raise RuntimeError("RQ2 crossed run did not pass; download the ZIP/log for review.")
    if mode == "calibration":
        print("Calibration completed. Download ZIP/log for review before starting the full panel.", flush=True)
        print("Calibration report:", folder / "report.json", flush=True)
    return folder


In [ ]:
verify_rq2_crossed_prerequisites()


## Execute exactly one mode

First run the seven-question calibration and review its ZIP/log. For the full30, preserve all calibration files and pass its report path. The old merge/no-merge v2 calibration cannot gate this new crossed experiment.

Later in the same live session: `run_rq2_crossed(mode="full", calibration_report=crossed_calibration_dir / "report.json")`.

Optional separate diagnostic: `run_rq2_crossed(mode="oracle")`. No new input ZIP or environment is needed. Every invocation loads weights once and reuses downloads; model memory is released when its subprocess exits.


In [ ]:
if RUN_CROSSED_CALIBRATION:
    crossed_calibration_dir = run_rq2_crossed(mode="calibration")
elif RUN_CROSSED_FULL:
    crossed_full_dir = run_rq2_crossed(mode="full", calibration_report=CALIBRATION_REPORT)
elif RUN_BROLIO_ORACLE:
    brolio_oracle_dir = run_rq2_crossed(mode="oracle")
else:
    raise RuntimeError("Enable one mode in the first cell.")


## Download before stopping the session

Keep the timestamped ZIP and matching console log. The runner records per-image source/basis/selection maps, each random selected set and seed, captured feature tuples, exact replay controls, fresh native repeats, answers and seed-wise interaction summaries. The full ZIP embeds the passed calibration evidence. These larger feature archives are ignored by Git; pushing code does not upload results.


In [ ]:
from IPython.display import FileLink, display
paths = [setup_log, runtime_report]
for folder in rq2_crossed_runs:
    paths.extend([folder.with_suffix(".zip"), folder.with_suffix(".log")])
for path in paths:
    if path.is_file():
        print(path)
        display(FileLink(str(path)))
